## 手撕系列

In [ ]:
# 手撕MHA
import torch 
import torch.nn as nn
import torch.nn.functional as F # 用于计算softmax
import math 

# 定义MHA类
class MultiHeadAttention(nn.Module):
    # 初始化变量：d_model为模型维度（q、k、v维度）；num_heads为头数
    def __init__(self, d_model, num_heads):
        # 父类声明
        super().__init__()
        # 注意保证维度可以整除头数
        assert d_model % num_heads == 0
        # 定义关键变量
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads # 每个头的维度H
        # 初始化四个线性层，分别用于q、k、v和输出
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        # 拿到输入的batch_size和seq_len
        batch_size, seq_len, _ = x.shape
        # 1. 线性变换与多头拆分，维度变化为:[batch_size, seq_len, d_model] -> [batch_size, seq_len, num_heads, head_dim] -> [batch_size, num_heads, seq_len, head_dim]
        q = self.w_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.w_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.w_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 2.计算注意力分数，矩阵点积，q：[b, nh, l, hd] k^T:[b, nh, hd, l] -> attn_scores:[b, nh, l, l]
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 3.是否加mask
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9) # 将0替换为极小值，softmax之后为0

        # 4.softmax，attn_prob:[b, nh, l, l]
        attn_prob = F.softmax(attn_scores, dim = -1)

        # 5.加权求和,output:[b, nh, l, hd]
        output = torch.matmul(attn_scores, v)

        # 6.多头合并,[b, nh, l, hd] -> [b, l, nh, hd] -> [b, l, d]
        output = output.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.w_o(output)
        
def generate_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

d_model = 128
num_heads = 8
mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(2, 5, 128)
mask = generate_causal_mask(5)
print(mask)
# 前向传播
output = mha(x, mask=mask)
# 打印维度
print(x.shape)
print(output.shape)
print(mask.shape)
print(output)

In [4]:
# 手搓qwen3，run_qwen3.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import argparse
from transformers import AutoTokenizer, AutoConfig

def load_weights_from_hf(model:nn.Module, model_path:str, device:torch.device, dtype:torch.dtype):
    model.to(device=device, dtype=dtype)

class Qwen3CausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        pass

def main():
    parser = argparse.ArgumentParser(description="手搓qwen3")
    parser.add_argument("--model", type=str, default="Qwen/Qwen3-8B")
    parser.add_argument("--prompt", type=str, default="Who are u")
    parser.add_argument("--max-tokens", type=int, default=128)
    parser.add_argument("--temperature", type=float, default=0)
    parser.add_argument("--top-k", type=int, default=50)

    args = parser.parse_args()
    device = torch.device()
    dtype = torch.bfloat16

    # 读取config
    config = AutoConfig.from_pretrained(args.model)

    # 创建模型
    model = Qwen3CausalLM(config)

    # 加载权重
    load_weights_from_hf(model, args.model, device, dtype)
    model.eval()

    # 推理生成
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    messages = [{"role":"user", "content":args.prompt}]
    text = tokenizer.apply_chat_template(messages, tokenizer=False, add_generation_prompt=True, enable_thinking=False)
    print(text)

if __name__ == "__main__":
    main()

ImportError: cannot import name 'hf_api' from 'transformers.utils' (c:\Users\22122\anaconda3\envs\pytorch\Lib\site-packages\transformers\utils\__init__.py)